Загружаем библиотеки

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from tqdm import tqdm

Посмотрев структуру URL на сайте LifeHacker, заметим, что каждая страница имеет следующий формат:


https://lifehacker.ru/topics/technology/?page=N


Где N — номер страницы.Ссылка на первую страницу выглядит так:


https://lifehacker.ru/topics/technology/


Вторая страница:


https://lifehacker.ru/topics/technology/?page=2


И так далее.

Открываем страницу в браузере и смотрим на структуру HTML-разметки. Каждый блок статьи находится внутри элемента `<div>` с уникальным классом `.post-card__title-link`. Внутри этого блока расположен элемент <span class="post-card__title">, содержащий название статьи.

Сам текст статьи находится по уникальной ссылке на полную версию материала.

In [ ]:
response = requests.get('https://lifehacker.ru/topics/technology/')
response.text

'<!DOCTYPE html><html  lang="ru-RU"><head><meta charset="utf-8">\n<meta name="viewport" content="width=device-width, initial-scale=1">\n<title>Технологии — Лайфхакер</title>\n<link rel="preconnect" href="https://www.googletagmanager.com">\n<link rel="preconnect" href="https://top-fwz1.mail.ru">\n<link rel="preconnect" href="https://mc.yandex.ru">\n<link rel="preconnect" href="https://static.criteo.net">\n<link rel="preconnect" href="https://exchange.buzzoola.com">\n<link rel="preconnect" href="https://pb.adriver.ru">\n<link rel="preconnect" href="https://ad.mail.ru">\n<script src="https://yandex.ru/ads/system/context.js" crossorigin="anonymous" async></script>\n<script src="https://yandex.ru/ads/system/header-bidding.js" async></script>\n<style>@font-face{ascent-override:92%;descent-override:14%;font-family:Commissioner Fallback;size-adjust:102%;src:local("Trebuchet MS")}@font-face{font-display:swap;font-family:Commissioner;font-style:normal;font-weight:400;src:url(/_nuxt/commissioner-

Получаем контент первой страницы

In [ ]:
# Определяем базовый URL и общий шаблон пагинации
base_url = 'https://lifehacker.ru/topics/technology/'
response = requests.get('https://lifehacker.ru/topics/technology/') # получаем контент первой страиниц
soup = BeautifulSoup(response.text, 'lxml') # инициализируем объект bs4 и задаем парсер lxml

Ищем блоки с материалами на странице

In [ ]:
soup.find_all('a', class_='lh-small-article-card__link')

[<a aria-label="Найди, подумай, исследуй: как работают функции чат-ботов и когда их включать" class="lh-small-article-card__link" data-jest="link" href="/funkcii-chat-botov/" title="Найди, подумай, исследуй: как работают функции чат-ботов и когда их включать"><!--[--><!--]--></a>,
 <a aria-label="Xiaomi показала смартфон Redmi K80 Ultra с батареей 7 410 мА⋅ч и мощными динамиками" class="lh-small-article-card__link" data-jest="link" href="/anons-redmi-k80-ultra/" title="Xiaomi показала смартфон Redmi K80 Ultra с батареей 7 410 мА⋅ч и мощными динамиками"><!--[--><!--]--></a>,
 <a aria-label="Xiaomi представила&amp;nbsp;компактный планшет Redmi K Pad и 12,5‑дюймовый Pad 7S&amp;nbsp;Pro" class="lh-small-article-card__link" data-jest="link" href="/anons-redmi-k-pad-i-xiaomi-pad-7s-pro/" title="Xiaomi представила&amp;nbsp;компактный планшет Redmi K Pad и 12,5‑дюймовый Pad 7S&amp;nbsp;Pro"><!--[--><!--]--></a>,
 <a aria-label="Garmin представила трекер сна, который надевается на плечо и работ

Собираем ссылки на материалы

In [ ]:
raw_items = soup.find_all('a', class_='lh-small-article-card__link')  # ищем все a-элементы с классом lh-small-article-card__link
links = [f"https://lifehacker.ru{item.get('href')}" for item in raw_items] # Извлекаем href-атрибуты и сразу формируем полные ссылки
# Проверим наличие ссылок
print(f"Найдено {len(links)} ссылок на статьи.")

Найдено 30 ссылок на статьи.


Получили ссылки на 30 статей с первой страницы, осталось собрать ссылки со всех 10 страниц

In [ ]:
# Кол-во страниц для парсинга
num_pages = 10

In [ ]:
# Массив для хранения ссылок
parsed_urls = []

In [ ]:
# Пробежимся по страницам и соберём ссылки
for page in range(1, num_pages + 1):
    if page == 1:
        url = base_url
    else:
        url = f"{base_url}?page={page}"

    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'lxml')

    # Находим все ссылки на статьи
    raw_items = soup.find_all('a', class_='lh-small-article-card__link')

    # Формируем полные ссылки
    links = [f"https://lifehacker.ru{item.get('href')}" for item in raw_items]

    # Добавляем ссылки в список parsed_urls
    parsed_urls.extend(links)


In [ ]:
# Проверим размер полученного списка
print(f"Всего получено {len(parsed_urls)} ссылок.")

Всего получено 300 ссылок.


In [ ]:
# Проверим, что ссылки собраны правильно
print(parsed_urls[:5])  # показываем первые 5 ссылок

['https://lifehacker.ru/funkcii-chat-botov/', 'https://lifehacker.ru/anons-redmi-k80-ultra/', 'https://lifehacker.ru/anons-redmi-k-pad-i-xiaomi-pad-7s-pro/', 'https://lifehacker.ru/anons-garmin-index-sleep-monitor/', 'https://lifehacker.ru/anons-amazfit-active-2-square/']


Пройдемся по полученным ссылкам и спарсим сами статьи

In [ ]:
base_url = 'https://lifehacker.ru'

In [ ]:
url = 'https://lifehacker.ru/anons-garmin-index-sleep-monitor/'
response = requests.get(url)
soup = BeautifulSoup(response.text, 'lxml')

In [ ]:
content_block = soup.find('div', class_='post-content')

не получается загруить все статьи, всегда ломается только на середине((

In [ ]:
result = []

for url in tqdm(parsed_urls):
    article = {}

    # Получаем HTML статьи
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'lxml')

    # Заголовок статьи
    article['title'] = soup.find('h1', class_='article-card__title').text.strip()

    # Основной текст статьи
    # Content собирается из абзацев и других текстовых элементов
    paragraphs = soup.find_all(['p', 'ul', 'ol', 'blockquote'])
    article['text'] = "\n\n".join([element.text.strip() for element in paragraphs])

    # Дополнительные метаданные (например, автор или дата публикации)

    result.append(article)

 46%|████▌     | 137/300 [04:38<05:31,  2.03s/it]


ConnectionError: HTTPSConnectionPool(host='lifehacker.ruhttps', port=443): Max retries exceeded with url: /burninghut.ru/prompty-dlya-uprosheniya-zhizni/?utm_source=lifehacker.ru&utm_medium=referral&utm_campaign=teaser&utm_content=prompty-dlya-uprosheniya-zhizni (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7d0dd4f7ee50>: Failed to resolve 'lifehacker.ruhttps' ([Errno -2] Name or service not known)"))

Создаем DataFrame

In [ ]:
data = pd.DataFrame(result)

Ограничим ширину отображаемых колонок для удобства просмотра

In [ ]:
pd.set_option('display.max_colwidth', 400)

Просмотр записей

In [ ]:
data

,title,text
0,"Найди, подумай, исследуй: как работают функции чат-ботов и когда их включать","Автор Лайфхакера\n\nChatGPT, DeepSeek и прочие чат-боты устроены максимально просто: общаться с ними интуитивно сможет каждый, кто когда-нибудь пользовался любым мессенджером или поисковиком. Но помимо окна ввода текста боты предлагают ещё один элемент: переключатели дополнительных опций, у которых обычно нет вменяемого описания. Например, в детище OpenAI это выглядит так:\n\nА в китайском Qwe..."
1,Xiaomi показала смартфон Redmi K80 Ultra с батареей 7 410 мА⋅ч и мощными динамиками,"Автор Лайфхакера\n\nXiaomi представила мощный смартфон Redmi K80 Ultra — преемника Redmi K70 Extreme Edition и нового конкурента доступных Android-флагманов, таких как Honor GT Pro. Несмотря на премиальные характеристики, компании удалось сохранить привлекательную цену.\n\nТелефон оснащён 6,83-дюймовым AMOLED-дисплеем с разрешением 2К, частотой обновления 144 Гц и пиковой яркостью 3 200 нит. Д..."
2,"Xiaomi представила компактный планшет Redmi K Pad и 12,5‑дюймовый Pad 7S Pro","Автор Лайфхакера\n\nXiaomi представила планшеты Redmi K Pad и Xiaomi Pad 7S Pro. Первый — компактный 8,8‑дюймовый гаджет, который позиционируется как конкурент iPad mini 7, Lenovo Legion Y700 и Samsung Galaxy Tab S11. Второй уже покрупнее — он оснащён 12,5‑дюймовым дисплеем.\n\nRedmi K Pad получил IPS-экран с разрешением 3K, частотой обновления 165 Гц и максимальной яркостью 700 нит. Планшет р..."
3,"Garmin представила трекер сна, который надевается на плечо и работает без подзарядки неделю","Автор Лайфхакера\n\nGarmin выпустила устройство Index Sleep Monitor — повязку, предназначенную для отслеживания сна. В отличие от традиционных фитнес-браслетов и часов, новинка крепится на верхнюю часть плеча, благодаря чему не мешает во время сна и позволяет оставить часы на зарядке.\n\nIndex Sleep Monitor фиксирует фазы сна (лёгкий, глубокий и REM), частоту сердцебиения, дыхание и температур..."
4,Представлены часы Amazfit Active 2 Square с AMOLED-дисплеем и 10-дневной автономностью,"Автор Лайфхакера\n\nAmazfit представила умные часы Active 2 Square с ярким дисплеем и впечатляющим временем автономной работы. Устройство доступно с разными вариантами ремешков, которые позволяют менять стиль от спортивного до делового.\n\nНовинка отличается от стандартной версии Active 2 квадратным 1,75-дюймовым экраном с закруглёнными углами. Разрешение дисплея составляет 390 × 450 пикселей,..."
...,...,...
132,"Вместо тысячи адаптеров: Huawei, Honor, Oppo и Vivo будут использовать единый стандарт зарядки","Автор Лайфхакера\n\nЧетыре крупнейших китайских производителя смартфонов — Huawei, Honor, Oppo и Vivo — договорились о поддержке единого стандарта быстрой зарядки UFCS 2.0. Компании подписали соглашение о взаимной авторизации технологии, что стало важным шагом к устранению фрагментации в экосистеме зарядных устройств.\n\nНовый протокол UFCS 2.0 был представлен на конференции UFCS Industry Deve..."
133,Нейросети в руках детей: 11 крутых применений кроме решения домашки,"Автор Лайфхакера\n\nПока взрослые пытаются применять нейросети для рабочих задач и автоматизации рутины, дети используют их совсем иначе — свободно, играючи и неожиданно изобретательно. Да, конечно, самый популярный сценарий — сделать домашнее задание. Но помимо этого детский ум находит ChatGPT и его собратьям немало других применений: от комиксов и музыки до игр, обучения и самовыражения.\n\n..."
134,OnePlus представила недорогие TWS-наушники Buds 4 с шумоподавлением до 55 дБ,"Автор Лайфхакера\n\nOnePlus представила недорогие TWS-наушники Buds 4. Новинка получила активное шумоподавление с эффективностью до 55 дБ и поддерживает ультраширокополосный режим в диапазоне до 5,5 кГц. Также есть система из трёх микрофонов, дополненных системой шумоподавления для голосовых вызовов на базе ИИ. По заявлениям производителя, технология обеспечивает высокую чёткость речи даже в ш..."
135,"12 приложений для Android, которые прокачают камеру вашего смартфона